<h1>Dataset Load & Initial Inspection</h1>

In [10]:
import pandas as pd
import numpy as np
import re

df_raw = pd.read_csv('raw_data.csv')

print(f"Total Raw Records Loaded: {len(df_raw)}")
print("\n--- Data Information ---")
df_raw.info()

print("\n--- Raw Data Preview ---")
display(df_raw.head(10))

Total Raw Records Loaded: 52

--- Data Information ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Brand                52 non-null     object
 1   Category             52 non-null     object
 2   Product_Name         52 non-null     object
 3   Price_Raw            50 non-null     object
 4   Target_Demographic   52 non-null     object
 5   Availability_Status  52 non-null     object
 6   Source_URL           52 non-null     object
dtypes: object(7)
memory usage: 3.0+ KB

--- Raw Data Preview ---


,Brand,Category,Product_Name,Price_Raw,Target_Demographic,Availability_Status,Source_URL
0,Nike,Men's Graphic T-Shirts,Nike Sportswear Club Fleece Tee,"₹ 2,860","Athletes, Fitness Users & Streetwear Enthusiasts",In Stock,https://www.nike.com/in/
1,Nike,Performance Footwear,Nike Dri-FIT Legend Fitness Tee,"₹ 7,760","Athletes, Fitness Users & Streetwear Enthusiasts",In Stock,https://www.nike.com/in/
2,Nike,Sportswear Tops,Nike Max90 Heavyweight Graphic Tee,"₹ 9,440","Athletes, Fitness Users & Streetwear Enthusiasts",In Stock,https://www.nike.com/in/
3,Nike,Hoodies & Sweatshirts,Nike Pro Warm Tight,"₹ 3,780","Athletes, Fitness Users & Streetwear Enthusiasts",In Stock,https://www.nike.com/in/
4,Nike,Training Tights,Nike Pegasus 40 Running Shoes,"₹ 10,550","Athletes, Fitness Users & Streetwear Enthusiasts",In Stock,https://www.nike.com/in/
5,Nike,Men's Graphic T-Shirts,Nike Court Vision Low,"₹ 10,070","Athletes, Fitness Users & Streetwear Enthusiasts",In Stock,https://www.nike.com/in/
6,Nike,Performance Footwear,Nike Sportswear Essential Hoodie,"₹ 7,420","Athletes, Fitness Users & Streetwear Enthusiasts",In Stock,https://www.nike.com/in/
7,Nike,Sportswear Tops,Nike Element Half-Zip,NaN,"Athletes, Fitness Users & Streetwear Enthusiasts",Out of Stock,https://www.nike.com/in/
8,Nike,Hoodies & Sweatshirts,Nike Dri-FIT Academy Top,"₹ 9,180","Athletes, Fitness Users & Streetwear Enthusiasts",In Stock,https://www.nike.com/in/
9,Nike,Training Tights,Nike SB Logo Tee,"₹ 1,560","Athletes, Fitness Users & Streetwear Enthusiasts",In Stock,https://www.nike.com/in/


<h2>Step 1 & 2 - String Cleaning, Standardization & Deduplication</h2>

In [5]:
# 1. Stripping unwanted spaces from text columns
text_columns = ['Brand', 'Category', 'Product_Name', 'Target_Demographic', 'Price_Raw']
for col in text_columns:
    df_raw[col] = df_raw[col].astype(str).str.strip()

# 2. Standardizing Brand Names
df_raw['Brand'] = df_raw['Brand'].replace({'H & M': 'H&M'})

# 3. Duplicate Detection & Removal
duplicate_count = df_raw.duplicated().sum()
print(f"Duplicate Rows Found: {duplicate_count}")

df_cleaned = df_raw.drop_duplicates().copy()
print(f"Dataset size after removing duplicates: {len(df_cleaned)} rows")

Duplicate Rows Found: 2
Dataset size after removing duplicates: 50 rows


<h2>Step 3 - Regex Parsing for Raw Prices & Handling Missing Values</h2>

In [11]:
# Function to parse price strings into clean numbers
def parse_price(price_str):
    if pd.isna(price_str) or str(price_str).upper() in ['N/A', '-', 'UNKNOWN', 'NAN']:
        return np.nan
    numbers = re.findall(r'\d+', str(price_str).replace(',', ''))
    if numbers:
        return float(numbers[0])
    return np.nan

# Extract Numeric Price Column
df_cleaned['Price_INR'] = df_cleaned['Price_Raw'].apply(parse_price)

# Check missing price counts before imputation
missing_prices = df_cleaned['Price_INR'].isna().sum()
print(f"Missing Price Entries Detected: {missing_prices}")
display(df_cleaned[df_cleaned['Price_INR'].isna()])

Missing Price Entries Detected: 2


,Brand,Category,Product_Name,Price_Raw,Target_Demographic,Availability_Status,Source_URL,Price_INR
7,Nike,Sportswear Tops,Nike Element Half-Zip,nan,"Athletes, Fitness Users & Streetwear Enthusiasts",Out of Stock,https://www.nike.com/in/,NaN
44,UNIQLO,Minimalist Outerwear,UNIQLO Ultra Light Down Jacket,nan,"Broad Demographic, Students & Minimalist Adults",Out of Stock,https://www.uniqlo.com/in/,NaN


<h2>Step 4 - Group-Wise Median Imputation & Statistical Aggregation</h2>

In [14]:
# Missing prices
df_cleaned['Price_INR'] = df_cleaned.groupby('Brand')['Price_INR'].transform(
    lambda group: group.fillna(group.median())
)

# Brand-level Price Summary Table
brand_summary = df_cleaned.groupby('Brand')['Price_INR'].agg(
    Total_Products='count',
    Min_Price_INR='min',
    Max_Price_INR='max',
    Avg_Price_INR='mean',
    Median_Price_INR='median'
).reset_index()

brand_summary['Avg_Price_INR'] = brand_summary['Avg_Price_INR'].round(2)

print("=== BRAND-LEVEL PRICE SUMMARY TABLE ===")
display(brand_summary)

=== BRAND-LEVEL PRICE SUMMARY TABLE ===


,Brand,Total_Products,Min_Price_INR,Max_Price_INR,Avg_Price_INR,Median_Price_INR
0,H&M,10,610.0,2470.0,1560.0,1875.0
1,Nike,10,1560.0,10550.0,7038.0,7760.0
2,UNIQLO,10,1020.0,4850.0,3076.0,3820.0
3,Zara,10,3320.0,8380.0,5980.0,5970.0
4,adidas,10,1410.0,9310.0,6031.0,6625.0


<h2>Step 5 - Data Export</h2>

In [13]:
# Final cleaned datasets
df_cleaned.to_csv('cleaned_data.csv', index=False)
brand_summary.to_csv('brand_summary.csv', index=False)

print("Data Cleaning Workflow Executed Successfully!")
print("Output files generated: 'cleaned_data.csv' & 'brand_summary.csv'")

Data Cleaning Workflow Executed Successfully!
Output files generated: 'cleaned_data.csv' & 'brand_summary.csv'
